In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import TfidfTransformer

# Initialize a matrix for the 200x200 grid (40,000 cells) and 85 POI categories
# Use mesh_x and mesh_y (1-200) to create a flattened index
poi_matrix = np.zeros((40000, 85))
directory = "cell_POIcat.csv/"

for filename in os.listdir(directory):
    try:
        parts = filename.split(',')
        if len(parts) == 4:
            x, y, cat_id, count = map(int, parts)
            # Flatten 2D grid (1-200) to 1D index (0-39999)
            grid_idx = (x - 1) * 200 + (y - 1)
            poi_matrix[grid_idx, cat_id - 1] = count
    except ValueError:
        continue

poi_df = pd.DataFrame(poi_matrix)

In [2]:
tfidf = TfidfTransformer()
weighted_poi = tfidf.fit_transform(poi_df)

# LDA for Functional Zones
# Choose 5 zones: e.g., Residential, Business, Industrial, Transit, Green Space
lda = LatentDirichletAllocation(n_components = 5, random_state=42)
zone_distributions = lda.fit_transform(weighted_poi)

# Save the results back to a lookup table
zone_cols = [f'zone_{i}_prob' for i in range(5)]
dist_df = pd.DataFrame(zone_distributions, columns=zone_cols)
poi_df = pd.concat([poi_df, dist_df], axis=1)

poi_df['grid_id'] = range(40000)

feature_names = [f"Cat_{i}" for i in range(1, 86)]
for topic_idx, topic in enumerate(lda.components_):
    top_features = [feature_names[i] for i in topic.argsort()[:-6:-1]]
    print(f"Zone {topic_idx} Top POIs: {top_features}")


Zone 0 Top POIs: ['Cat_48', 'Cat_54', 'Cat_59', 'Cat_47', 'Cat_82']
Zone 1 Top POIs: ['Cat_59', 'Cat_69', 'Cat_81', 'Cat_62', 'Cat_48']
Zone 2 Top POIs: ['Cat_79', 'Cat_80', 'Cat_63', 'Cat_60', 'Cat_81']
Zone 3 Top POIs: ['Cat_81', 'Cat_84', 'Cat_75', 'Cat_76', 'Cat_79']
Zone 4 Top POIs: ['Cat_74', 'Cat_60', 'Cat_73', 'Cat_47', 'Cat_48']
